# Phantom Ink Question Generator

AI 驅動的「靈媒遊戲」題目生成器

## 流程
1. **出題** — AI 扮演出題老師，產生七道由難到易的問答
2. **驗題** — AI 扮演驗題老師，檢查題目品質
3. **模擬** — AI 扮演玩家，逐題模擬猜測過程
4. **注音轉換** — 自動將回答轉換為注音符號

## 第一步：安裝相依套件

In [ ]:
# @title 安裝相依套件
!pip install huggingface_hub pypinyin pydantic zhconv groq -q

## 第二步：設定 API Key

### 選項 A：Groq（推薦）
免費額度較高，生成速度快。

**申請 Key：**
1. 到 https://console.groq.com/keys 註冊/登入
2. 按 **Create API Key**，複製 Key（開頭是 `gsk_`）

### 選項 B：Hugging Face
免費，免下載模型。

**申請 Token：**
1. 到 https://huggingface.co/settings/tokens 註冊/登入
2. 按 **Create new token** → 選 **Read** 權限
3. 複製 Token（開頭是 `hf_`）

**在下方選擇後端、貼上 Key 即可使用。**

In [ ]:
# @title 選擇後端 + 輸入 API Key + 模型

import os

# 選擇 LLM 後端
BACKEND = "Groq"  # @param ["Groq", "Hugging Face"]

# API Key（根據後端貼上對應的 Key）
API_KEY = ""  # @param {type: "string"}

# 模型名稱（可視後端調整，留空=使用預設）
MODEL_NAME = ""  # @param {type: "string"}

if BACKEND == "Groq":
    os.environ["GROQ_API_KEY"] = API_KEY
    print(f"✅ Groq API Key 已設定（前8碼：{API_KEY[:8]}...）")
    model_display = MODEL_NAME or "llama-3.3-70b-versatile（預設）"
else:
    os.environ["HF_TOKEN"] = API_KEY
    os.environ["HF_MODEL"] = MODEL_NAME or "Qwen/Qwen2.5-7B-Instruct"
    print(f"✅ HF Token 已設定（前8碼：{API_KEY[:8]}...）")
    model_display = MODEL_NAME or "Qwen/Qwen2.5-7B-Instruct（預設）"

print(f"✅ 後端：{BACKEND}")
print(f"✅ 模型：{model_display}")

## 第三步：從 GitHub 下載模組

Colab 無法直接引用本機 .py 檔，這裡直接從 GitHub Raw 抓取所有必要模組。

In [ ]:
# @title 下載模組 + 重新載入

import os
import sys
import urllib.request
import urllib.error

# 清除已載入的模組快取，確保 import 拿到最新檔案
for mod in list(sys.modules.keys()):
    if mod in ('generator', 'backends', 'models', 'prompts', 'bopomofo', 'utils', 'game'):
        del sys.modules[mod]

# ===== 版本設定 =====
# 修改這裡的 TAG 來鎖定版本，例：VERSION_TAG = "v1.0"
# 如果 TAG 找不到某個檔案，會自動降級到 master
VERSION_TAG = "v0.3"  # @param {type: "string"}

GITHUB_BASE = "https://raw.githubusercontent.com/csongs/Phantom-Ink-Question"
FILES = ["backends.py", "models.py", "prompts.py", "bopomofo.py", "generator.py", "utils.py", "game.py"]

# 先試 TAG，再試 master
refs = [f"{GITHUB_BASE}/{VERSION_TAG}", f"{GITHUB_BASE}/master"]
ref_names = [VERSION_TAG, "master"]

for f in FILES:
    downloaded = False
    for ref_url, ref_name in zip(refs, ref_names):
        url = f"{ref_url}/{f}"
        try:
            urllib.request.urlretrieve(url, f)
            print(f"  ✓ {f}（{ref_name}）")
            downloaded = True
            break
        except urllib.error.HTTPError as e:
            if e.code == 404:
                continue  # 404 → 換下一個 ref
            print(f"  ✗ {f} HTTP {e.code}")
            break
    if not downloaded:
        ref_names_str = "、".join(ref_names)
        print(f"  ✗ {f} 所有 ref 都找不到（{ref_names_str}）")

# 匯入模組
from generator import PhantomInkGenerator
from models import QuestionSetWithMeta, QuestionItem
from bopomofo import to_bopomofo, to_bopomofo_cells, reveal_bopomofo, count_bopomofo_cells
from utils import save_question_set, load_question_set, export_to_colab_json, print_question_set
from game import play_game, play_colab_game

print(f"\n✅ 全部模組載入完成（版本：{VERSION_TAG}）")

## 第四步：單題生成

輸入一個謎底，自動完成出題 → 驗題 → 模擬玩家

In [ ]:
# @title 單題生成

ANSWER = ""  # @param {type: "string"}
ANSWER_MODE = "AI 填回答"  # @param ["AI 填回答", "人自己想"]
NUM_QUESTIONS = 7  # @param {type: "integer"}
SKIP_REVIEW = False  # @param {type: "boolean"}
SKIP_SIMULATION = True  # @param {type: "boolean"}
MAX_RETRIES = 3  # @param {type: "integer"}

# 決定 backend 字串
backend_type = "groq" if BACKEND == "Groq" else "hf"
api_key = os.environ.get("GROQ_API_KEY" if backend_type == "groq" else "HF_TOKEN", "")
model_name = MODEL_NAME or ""

gen = PhantomInkGenerator(
    token=api_key,
    model=model_name,
    max_retries=MAX_RETRIES,
    backend=backend_type,
)

result = gen.generate(
    answer=ANSWER,
    skip_review=SKIP_REVIEW,
    skip_simulation=SKIP_SIMULATION,
    verbose=True,
    answer_mode=("ai" if ANSWER_MODE == "AI 填回答" else "human"),
    num_questions=NUM_QUESTIONS,
)

print_question_set(result)

## （此步驟已移除）

In [ ]:
# 此步驟已移除

## 第六步：匯出題庫

將生成的題組儲存為 JSON 或 CSV 格式

In [ ]:
# @title 匯出結果

EXPORT_FORMAT = "json"  # @param ["json", "csv"]
DOWNLOAD = False  # @param {type: "boolean"}

import datetime
import csv
import os

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

if 'result' not in locals() or not result.questions:
    print("⚠️ 請先執行「單題生成」")
else:
    if EXPORT_FORMAT == "json":
        filename = f"phantom_ink_{result.answer}_{timestamp}.json"
        save_question_set(result, filename, format="json")
        export_json = export_to_colab_json(result, f"game_{result.answer}_{timestamp}.json")
        print(f"✅ 已匯出：{filename}")
        print(f"✅ 遊戲用：game_{result.answer}_{timestamp}.json")
    else:
        filename = f"phantom_ink_{result.answer}_{timestamp}.csv"
        with open(filename, "w", encoding="utf-8-sig", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["answer", "question_number", "question", "reply"])
            for i, q in enumerate(result.questions):
                writer.writerow([result.answer, i + 1, q.question, q.reply])
        print(f"✅ 已匯出：{filename}")

    # 手動勾選才下載
    if DOWNLOAD:
        try:
            from google.colab import files
            files.download(filename)
            print("📥 已觸發下載")
        except:
            pass

## 第七步：注音墨水預覽

預覽每個回答的注音逐格顯示效果

In [ ]:
# @title 注音墨水預覽

if 'result' in locals() and result.questions:
    print(f"\n🎯 謎底：{result.answer}")
    print("=" * 50)

    for i, q in enumerate(result.questions):
        cells = to_bopomofo_cells(q.reply)
        total = len(cells)
        print(f"\nQ{i+1}. {q.question}")
        print(f"回答：{q.reply}")
        print(f"注音：{' '.join(cells)}")
        print(f"格數：{total}")

        print("揭露過程：")
        for step in range(1, total + 1):
            revealed = reveal_bopomofo(q.reply, step)
            pct = int(step / total * 100)
            bar = "█" * (pct // 10) + "░" * (10 - pct // 10)
            print(f"  [{bar}] {revealed}")

    print("\n✅ 預覽完成")
else:
    print("⚠️ 請先執行「單題生成」")

## 第八步：靈媒挑戰試玩

以 HTML/JS 渲染的互動遊戲畫面，在 Colab 中直接遊玩。

- **[🖋 顯示墨水]**：每按一次顯示回答的一個注音/聲調，墨水 +1
- **[➡ 下一題]**：覺得夠了就到下一題，先前問答鎖定
- **[🎯 提交謎底]**：隨時作答，答錯墨水 +3，不限次數
- **[👁 老天有眼]**：第5題開始獲得第一次，第7題按[完成線索]獲得第二次，可選過往任一答案多揭露一格
- **[📜 完成線索]**：第7題揭露完後出現

In [ ]:
# @title 靈媒遊戲試玩

if 'result' not in locals() or not result.questions:
    print("⚠️ 請先執行「單題生成」")
else:
    play_colab_game(result)

## 除錯工具

直接測試注音轉換功能

In [ ]:
# @title 注音轉換測試

TEST_WORDS = ["樂器行", "演奏廳", "鍵盤樂器", "木頭", "颱風", "腳踏車"]

print("測試注音轉換\n")
print(f"{'詞語':<10} {'注音':<30} {'格數':<6}")
print("-" * 50)

for word in TEST_WORDS:
    bpmf = to_bopomofo(word)
    cells = count_bopomofo_cells(word)
    print(f"{word:<10} {bpmf:<30} {cells:<6}")